# EoMT Fine-tuning for Anomaly Segmentation
## Day 1-2: Logit Normalization Loss + Head Fine-tuning

**Goal**: Fine-tune only the prediction head with Logit Normalization loss to improve anomaly segmentation.

**Strategy**:
- Freeze ViT encoder + EoMT decoder
- Train only mask/class prediction heads
- Use Automatic Mixed Precision (AMP) for speed
- Target: +1-2% AUPRC improvement

## Setup Environment

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('✓ Drive mounted')
    IS_COLAB = True
except:
    print('Not in Colab or Drive already mounted')
    IS_COLAB = False

## (OPTIONAL) Download Cityscapes Directly to Colab

If you have direct download links from Cityscapes, use this cell to download directly without Drive.


In [ ]:
# OPTION A: Direct download with links from Cityscapes website
# 1. Go to https://www.cityscapes-dataset.com/downloads/
# 2. Login and get your personal download links
# 3. Replace the URLs below:

DOWNLOAD_DIRECTLY = False  # Set to True if you have direct links

if DOWNLOAD_DIRECTLY:
    import subprocess
    
    # Replace with your actual links from Cityscapes
    LEFTIMG_URL = "YOUR_LEFTIMG_LINK_HERE"
    GTFINE_URL = "YOUR_GTFINE_LINK_HERE"
    
    dataset_dir = "/content/dataset"
    
    # Create directory
    !mkdir -p {dataset_dir}
    
    # Download using aria2c (faster with parallel connections)
    print("Downloading leftImg8bit (this takes ~30-45 min)...")
    !aria2c -x 16 -k 1M "{LEFTIMG_URL}" -d {dataset_dir}
    
    print("\nDownloading gtFine (this takes ~5 min)...")
    !aria2c -x 16 -k 1M "{GTFINE_URL}" -d {dataset_dir}
    
    print("\n✓ Download complete!")
    print(f"  Files saved to: {dataset_dir}")

else:
    print("""
    ℹ️  DIRECT DOWNLOAD DISABLED
    
    To download directly on Colab:
    1. Visit: https://www.cityscapes-dataset.com/downloads/
    2. Login with your account
    3. Get the direct download links (right-click → Copy Link)
    4. Paste the links in this cell:
       - LEFTIMG_URL = "..."
       - GTFINE_URL = "..."
    5. Set DOWNLOAD_DIRECTLY = True
    6. Run this cell
    
    Time estimates:
    - leftImg8bit (55 GB): ~30-45 min with aria2c parallel download
    - gtFine (241 MB): ~5 min
    - Total: ~1 hour
    
    Alternative: Upload to Drive (browser → slower but simpler)
    """)


In [ ]:
# Install dependencies
!pip install -q lightning scikit-learn torch torchvision timm

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
import os
from pathlib import Path

# Paths
if IS_COLAB:
    REPO_BASE = Path("/content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject ")
else:
    REPO_BASE = Path("..")

# Change to repo directory
os.chdir(REPO_BASE / "eomt")
print(f"Working directory: {os.getcwd()}")

# Training config
CONFIG = {
    # Model
    'pretrained_ckpt': str(REPO_BASE / "checkpoints" / "eomt_cityscapes.bin"),
    
    # Training
    'epochs': 2,
    'batch_size': 1,  # ← REDUCED: Use batch_size=1
    'gradient_accumulation_steps': 4,  # ← NEW: Accumulate over 4 steps = effective batch_size=4
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    
    # Freezing strategy
    'freeze_encoder': True,
    'freeze_decoder': True,
    'train_head_only': True,
    
    # Loss
    'use_logit_norm': True,
    'logit_norm_temp': 0.07,
    
    # AMP
    'use_amp': True,
    
    # Data - KEEP 1024x1024 (checkpoint compatible)
    'img_size': 1024,  # ← KEEP: Must match checkpoint
    'cityscapes_root': str(REPO_BASE / "dataset" / "cityscapes"),
    'num_workers': 2,
    
    # Output
    'output_dir': str(REPO_BASE / "checkpoints" / "fine_tuned"),
    'save_every': 1,
}

# Create output directory
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)

print("✓ Configuration loaded")
print(f"  Image size: {CONFIG['img_size']}x{CONFIG['img_size']} (checkpoint compatible)")
print(f"  Batch size: {CONFIG['batch_size']}")
print(f"  Gradient accumulation: {CONFIG['gradient_accumulation_steps']}")
print(f"  Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"  Pretrained checkpoint: {CONFIG['pretrained_ckpt']}")
print(f"  Output directory: {CONFIG['output_dir']}")
print(f"  Freeze encoder: {CONFIG['freeze_encoder']}")
print(f"  Use AMP: {CONFIG['use_amp']}")
print(f"  Logit Normalization: {CONFIG['use_logit_norm']}")
print(f"\n💾 Estimated GPU memory: ~8-9 GB (safe for T4 14GB)")


## Import EoMT Components

In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm

# Add eomt to path
sys.path.insert(0, str(REPO_BASE / "eomt"))

from models.eomt import EoMT
from models.vit import ViT
from training.mask_classification_semantic import MaskClassificationSemantic

print("✓ Imports successful")

## Logit Normalization Loss Implementation

In [ ]:
class LogitNormalizationLoss(nn.Module):
    """
    Logit Normalization Loss for improving confidence calibration.
    
    Normalizes logits to unit sphere before computing cross-entropy.
    This prevents overconfident predictions and improves anomaly detection.
    
    Reference: https://arxiv.org/abs/2205.09310
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, logits, targets, mask=None, ignore_index=None):
        """
        Args:
            logits: [B, C, H, W] unnormalized logits
            targets: [B, H, W] ground truth labels
            mask: [B, H, W] optional mask for valid pixels
            ignore_index: optional int index to ignore (e.g., 255)
        """
        # Normalize logits to unit sphere
        logits_norm = F.normalize(logits, p=2, dim=1)  # L2 norm along channel dim
        
        # Scale by temperature
        logits_scaled = logits_norm / self.temperature
        
        # Cross-entropy with optional ignore_index
        if ignore_index is not None:
            loss = F.cross_entropy(
                logits_scaled,
                targets,
                reduction='none',
                ignore_index=ignore_index,
            )
        else:
            loss = F.cross_entropy(
                logits_scaled,
                targets,
                reduction='none'
            )
        
        # Apply mask if provided
        if mask is not None:
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1)
        else:
            loss = loss.mean()
        
        return loss

print("✓ Logit Normalization Loss implemented")


## Load Pre-trained Model

In [ ]:
# Model hyperparameters (match original training)
NUM_CLASSES = 19
MODEL_IMG_SIZE = (CONFIG['img_size'], CONFIG['img_size'])  # From config
PATCH_SIZE = 16
NUM_QUERIES = 100
NUM_BLOCKS = 3
BACKBONE_NAME = "vit_base_patch14_reg4_dinov2"

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Model input size: {MODEL_IMG_SIZE}")

# Build model
encoder = ViT(
    img_size=MODEL_IMG_SIZE,
    patch_size=PATCH_SIZE,
    backbone_name=BACKBONE_NAME,
)

network = EoMT(
    encoder=encoder,
    num_classes=NUM_CLASSES,
    num_q=NUM_QUERIES,
    num_blocks=NUM_BLOCKS,
    masked_attn_enabled=True,
)

model = MaskClassificationSemantic(
    network=network,
    img_size=MODEL_IMG_SIZE,
    num_classes=NUM_CLASSES,
    attn_mask_annealing_enabled=False,
    attn_mask_annealing_start_steps=None,
    attn_mask_annealing_end_steps=None,
    ckpt_path=CONFIG['pretrained_ckpt'],
    delta_weights=False,
    load_ckpt_class_head=True,
)

model = model.to(device)
print("✓ Model loaded with pre-trained weights")
print(f"  Positional embeddings: {MODEL_IMG_SIZE[0]//16}×{MODEL_IMG_SIZE[1]//16} = {(MODEL_IMG_SIZE[0]//16)**2} tokens")
print(f"  Memory: ~8-9 GB (batch_size={CONFIG['batch_size']}, img_size={CONFIG['img_size']})")


## Freeze Encoder/Decoder (Train Head Only)

In [ ]:
def freeze_module(module, freeze=True):
    """Freeze or unfreeze all parameters in a module"""
    for param in module.parameters():
        param.requires_grad = not freeze

# Freeze encoder (ViT backbone)
if CONFIG['freeze_encoder']:
    freeze_module(model.network.encoder, freeze=True)
    print("✓ Encoder frozen")

# Freeze decoder (Transformer blocks)
if CONFIG['freeze_decoder']:
    if hasattr(model.network, 'decoder'):
        freeze_module(model.network.decoder, freeze=True)
    print("✓ Decoder frozen")

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nParameter count:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"  Frozen: {total_params - trainable_params:,}")

## Setup Training Components

In [ ]:
# Loss function
if CONFIG['use_logit_norm']:
    criterion = LogitNormalizationLoss(temperature=CONFIG['logit_norm_temp'])
    print(f"✓ Using Logit Normalization Loss (T={CONFIG['logit_norm_temp']})")
else:
    criterion = nn.CrossEntropyLoss()
    print("✓ Using standard Cross-Entropy Loss")

# Optimizer (only trainable parameters)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)
print(f"✓ Optimizer: AdamW (lr={CONFIG['learning_rate']}, wd={CONFIG['weight_decay']})")

# AMP Scaler
if CONFIG['use_amp']:
    scaler = GradScaler()
    print("✓ AMP enabled (FP16 training)")
else:
    scaler = None
    print("✓ Full precision training (FP32)")

## Load Cityscapes Dataset

**Note**: You need to have Cityscapes dataset downloaded in the specified path.
If not available, this cell will show instructions to download it.

In [ ]:
# Check Cityscapes zip files (no extraction)
zip_dir = Path(CONFIG['cityscapes_root'])
left_zip = zip_dir / 'leftImg8bit_trainvaltest.zip'
right_zip = zip_dir / 'gtFine_trainvaltest.zip'

print(f"Zip directory: {zip_dir}")
print(f"  leftImg8bit_trainvaltest.zip: {'OK' if left_zip.exists() else 'MISSING'}")
print(f"  gtFine_trainvaltest.zip     : {'OK' if right_zip.exists() else 'MISSING'}")

if left_zip.exists() and right_zip.exists():
    print("\n✓ Cityscapes zips found — dataset ready for zip-based loading")
else:
    print("\n⚠️ Cityscapes zips not found")
    print("   Place both files in this folder:")
    print(f"   {zip_dir}")
    print("   Required files:")
    print("   - leftImg8bit_trainvaltest.zip")
    print("   - gtFine_trainvaltest.zip")


## Verify Cityscapes Zip Files

Use the official zip archives directly — no extraction will be performed.

In [ ]:
import subprocess
from pathlib import Path

# Use zip files directly (no extraction)
ZIP_DIR = Path(CONFIG['cityscapes_root'])
leftimg_zip = ZIP_DIR / 'leftImg8bit_trainvaltest.zip'
gtfine_zip = ZIP_DIR / 'gtFine_trainvaltest.zip'

print("Checking for Cityscapes zip files...")
print(f"  Zip directory: {ZIP_DIR}")
print(f"  leftImg8bit: {'OK' if leftimg_zip.exists() else 'MISSING'}")
print(f"  gtFine     : {'OK' if gtfine_zip.exists() else 'MISSING'}")

if leftimg_zip.exists() and gtfine_zip.exists():
    print("\n✓ Using zip files directly — no extraction needed")
    print("  This avoids Google Drive quota and speeds things up.")
else:
    print("\n⚠️ Zip files not found in the expected folder.")
    print("   Place both zips here:")
    print(f"   {ZIP_DIR}")
    print("   Files required:")
    print("   - leftImg8bit_trainvaltest.zip")
    print("   - gtFine_trainvaltest.zip")


## Build Cityscapes DataLoader (Zip-Based)

Load training dataset directly from `.zip` files using the project's datamodule.

In [ ]:
from pathlib import Path
from datasets.cityscapes_semantic import CityscapesSemantic

# Build datamodule from zip directory
zip_dir = Path(CONFIG['cityscapes_root'])
print(f"Using zip directory: {zip_dir}")
print(f"Image size: {CONFIG['img_size']}x{CONFIG['img_size']}")
print(f"Batch size: {CONFIG['batch_size']}")

try:
    dm = CityscapesSemantic(
        path=zip_dir,
        num_workers=CONFIG['num_workers'],
        batch_size=CONFIG['batch_size'],
        img_size=(CONFIG['img_size'], CONFIG['img_size']),
        num_classes=NUM_CLASSES,
        color_jitter_enabled=False,
        check_empty_targets=True,
    )
    dm.setup(stage='fit')
    train_loader = dm.train_dataloader()
    print(f"✓ DataLoader created from zips: {len(train_loader)} batches")
    print(f"  Batch size: {CONFIG['batch_size']}")
    print(f"  Image size: {CONFIG['img_size']}x{CONFIG['img_size']}")
    print(f"  Gradient accumulation steps: {CONFIG['gradient_accumulation_steps']}")
    print(f"  Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")
except Exception as e:
    print(f"⚠️  DataLoader creation failed: {str(e)}")
    print("  Ensure zip files exist:")
    print(f"  - {zip_dir / 'leftImg8bit_trainvaltest.zip'}")
    print(f"  - {zip_dir / 'gtFine_trainvaltest.zip'}")
    train_loader = None


## Initialize Training Loss Tracker

Save training metrics to CSV for visualization and analysis.

In [ ]:
import csv
from datetime import datetime

# Initialize loss tracker
loss_history = {
    'epoch': [],
    'batch': [],
    'loss': [],
    'timestamp': []
}

csv_path = Path(CONFIG['output_dir']) / 'training_loss.csv'

def log_loss(epoch, batch, loss_val):
    """Log training loss to CSV"""
    loss_history['epoch'].append(epoch)
    loss_history['batch'].append(batch)
    loss_history['loss'].append(float(loss_val))
    loss_history['timestamp'].append(datetime.now().isoformat())
    
    # Write to CSV (append mode)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'batch', 'loss', 'timestamp'])
        if f.tell() == 0:  # Write header if file is empty
            writer.writeheader()
        writer.writerow({
            'epoch': epoch,
            'batch': batch,
            'loss': float(loss_val),
            'timestamp': datetime.now().isoformat()
        })

print(f"✓ Loss tracker initialized")
print(f"  CSV path: {csv_path}")

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device, epoch):
    """
    Train for one epoch with AMP support, gradient accumulation, and zip-based dataloader.
    Converts instance targets to per-pixel labels and uses last-block logits.
    """
    model.train()
    total_loss = 0
    num_batches = len(dataloader)
    ignore_idx = getattr(model, 'ignore_idx', 255)
    accum_steps = CONFIG['gradient_accumulation_steps']

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    for batch_idx, batch in enumerate(pbar):
        # Dataloader yields (imgs, targets) from CityscapesSemantic
        images, targets = batch
        images = images.to(device)
        # targets is a list[dict]; convert to per-pixel [B, H, W]
        per_pixel_targets = model.to_per_pixel_targets_semantic(targets, ignore_idx)
        per_pixel_targets = torch.stack(per_pixel_targets).to(device).long()
        target_hw = per_pixel_targets.shape[-2:]
        
        # Diagnostic: check for invalid labels (first batch only)
        if batch_idx == 0:
            unique_labels = torch.unique(per_pixel_targets)
            invalid_labels = unique_labels[(unique_labels < 0) | ((unique_labels > 18) & (unique_labels != 255))]
            print(f"\n[Batch {batch_idx}] Target stats:")
            print(f"  Min: {per_pixel_targets.min().item()}, Max: {per_pixel_targets.max().item()}")
            print(f"  Unique labels: {unique_labels.tolist()}")
            if len(invalid_labels) > 0:
                print(f"  ⚠️ WARNING: Invalid labels found: {invalid_labels.tolist()}")
                print(f"     Valid range is [0-18] or {ignore_idx} (ignore)")
                print(f"     Clamping invalid labels to {ignore_idx}")
        
        # Safeguard: clamp any invalid labels to ignore_idx
        # Valid labels are 0-18 (19 classes) or 255 (ignore)
        # Anything else (e.g., -1 from license plate) should be ignored
        invalid_mask = (per_pixel_targets < 0) | ((per_pixel_targets > 18) & (per_pixel_targets != ignore_idx))
        if invalid_mask.any():
            per_pixel_targets = torch.where(invalid_mask, ignore_idx, per_pixel_targets)
        
        valid_mask = (per_pixel_targets != ignore_idx).to(device)

        if scaler is not None:
            # Prefer torch.amp.autocast; fallback to torch.cuda.amp.autocast
            try:
                with torch.amp.autocast('cuda'):
                    # Forward through network; use last block logits
                    mask_logits_list, class_logits_list = model(images)
                    mask_logits = mask_logits_list[-1]
                    class_logits = class_logits_list[-1]
                    per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
                    # Upsample logits to match target resolution
                    per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
                    loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx)
                    # Scale loss for gradient accumulation
                    loss = loss / accum_steps
            except AttributeError:
                with autocast(enabled=True):
                    mask_logits_list, class_logits_list = model(images)
                    mask_logits = mask_logits_list[-1]
                    class_logits = class_logits_list[-1]
                    per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
                    per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
                    loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx)
                    loss = loss / accum_steps

            scaler.scale(loss).backward()
        else:
            mask_logits_list, class_logits_list = model(images)
            mask_logits = mask_logits_list[-1]
            class_logits = class_logits_list[-1]
            per_pixel_logits = model.to_per_pixel_logits_semantic(mask_logits, class_logits)
            per_pixel_logits = F.interpolate(per_pixel_logits, size=target_hw, mode='bilinear', align_corners=False)
            loss = criterion(per_pixel_logits, per_pixel_targets, mask=valid_mask, ignore_index=ignore_idx)
            loss = loss / accum_steps
            loss.backward()

        # Update weights every accum_steps batches
        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == num_batches:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()

        # Undo scaling for logging
        total_loss += loss.item() * accum_steps
        avg_loss = total_loss / ((batch_idx + 1) // accum_steps + 1)

        # Log loss to CSV (every accumulation step)
        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == num_batches:
            log_loss(epoch, batch_idx // accum_steps, loss.item() * accum_steps)
            pbar.set_postfix({'loss': f'{loss.item() * accum_steps:.4f}'})

    return total_loss / num_batches

print("✓ Training function updated (ignore_index applied, spatial size aligned, AMP autocast fix)")


## Run Training

Main training loop with checkpoint saving.


In [ ]:
if train_loader is not None:
    print("Starting training...")
    print(f"Epochs: {CONFIG['epochs']}")
    print(f"Batches per epoch: {len(train_loader)}")
    print(f"Total batches: {CONFIG['epochs'] * len(train_loader)}")
    print()
    
    best_loss = float('inf')
    training_start = datetime.now()
    
    for epoch in range(CONFIG['epochs']):
        epoch_loss = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            epoch=epoch
        )
        
        print(f"\nEpoch {epoch+1}/{CONFIG['epochs']} - Loss: {epoch_loss:.4f}")
        
        # Save checkpoint every epoch
        if CONFIG['save_every'] > 0 and (epoch + 1) % CONFIG['save_every'] == 0:
            ckpt_path = Path(CONFIG['output_dir']) / f'eomt_finetuned_epoch{epoch+1}.pth'
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✓ Checkpoint saved: {ckpt_path}")
        
        # Save best model
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_ckpt_path = Path(CONFIG['output_dir']) / 'eomt_finetuned_best.pth'
            torch.save(model.state_dict(), best_ckpt_path)
            print(f"  ✓ Best model saved: {best_ckpt_path}")
    
    training_end = datetime.now()
    training_time = (training_end - training_start).total_seconds() / 3600
    
    print(f"\n{'='*50}")
    print(f"Training completed in {training_time:.1f} hours")
    print(f"Best loss: {best_loss:.4f}")
    print(f"Final loss: {epoch_loss:.4f}")
    print(f"{'='*50}")
    
    # Save final checkpoint
    final_ckpt = Path(CONFIG['output_dir']) / 'eomt_finetuned_final.pth'
    torch.save(model.state_dict(), final_ckpt)
    print(f"✓ Final checkpoint saved: {final_ckpt}")

else:
    print("⚠️  Cityscapes dataset not loaded. Cannot start training.")


## Quick Test: Forward Pass

Test that the model can do a forward pass with frozen weights.

## Plot Training Loss

Visualize training loss history after training completes.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Read CSV and plot
if csv_path.exists():
    df = pd.read_csv(csv_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Loss per batch
    ax1 = axes[0]
    for epoch in df['epoch'].unique():
        epoch_data = df[df['epoch'] == epoch]
        ax1.plot(epoch_data['batch'], epoch_data['loss'], label=f'Epoch {epoch+1}', marker='o', markersize=3)
    ax1.set_xlabel('Batch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss per Batch')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Epoch average loss
    ax2 = axes[1]
    epoch_loss = df.groupby('epoch')['loss'].mean()
    ax2.plot(epoch_loss.index, epoch_loss.values, marker='o', linewidth=2, markersize=8, color='red')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Average Loss')
    ax2.set_title('Average Loss per Epoch')
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(epoch_loss.index)
    
    plt.tight_layout()
    
    # Save figure
    plot_path = Path(CONFIG['output_dir']) / 'training_loss.png'
    plt.savefig(plot_path, dpi=100, bbox_inches='tight')
    print(f"✓ Loss plot saved: {plot_path}")
    plt.show()
else:
    print("⚠️  No training data available yet. Run training first!")


In [ ]:
# Create dummy input
dummy_input = torch.randn(1, 3, 1024, 1024).to(device)

print("Testing forward pass...")
model.eval()
with torch.no_grad():
    # Prefer torch.amp.autocast; fallback to torch.cuda.amp.autocast
    try:
        with torch.amp.autocast('cuda'):
            mask_logits_list, class_logits_list = model.network(dummy_input)
    except AttributeError:
        with autocast(enabled=CONFIG['use_amp']):
            mask_logits_list, class_logits_list = model.network(dummy_input)

print(f"✓ Forward pass successful")
print(f"  Mask logits: {mask_logits_list[-1].shape}")
print(f"  Class logits: {class_logits_list[-1].shape}")

# Clean up
del dummy_input, mask_logits_list, class_logits_list
torch.cuda.empty_cache()


## Next Steps

1. **Load Cityscapes Dataset**: Implement DataLoader for Cityscapes training set
2. **Run Training**: Execute training loop for 1-2 epochs
3. **Save Checkpoint**: Save fine-tuned model
4. **Evaluate**: Run evaluation on anomaly datasets using `evalAnomaly.py`
5. **Compare**: Compare results with baseline (MaxEntropy @ T=0.75)

**Expected improvement**: +1-2% AUPRC on average

## Evaluate Fine-tuned Model

Run evaluation on anomaly detection datasets.


In [ ]:
import subprocess

# Evaluation configuration
EVAL_METHODS = ['maxentropy']  # Best method from baseline
EVAL_TEMPERATURES = [0.75]      # Best temperature from baseline
EVAL_CHECKPOINT = str(Path(CONFIG['output_dir']) / 'eomt_finetuned_best.pth')

print("Running evaluation on anomaly datasets...")
print(f"Checkpoint: {EVAL_CHECKPOINT}")
print()

results_eval = []

for method in EVAL_METHODS:
    for temperature in EVAL_TEMPERATURES:
        print(f"Evaluating {method} at T={temperature}...")
        
        cmd = [
            'python', 'evalAnomaly.py',
            '--method', method,
            '--temperature', str(temperature),
            '--checkpoint', EVAL_CHECKPOINT,
        ]
        
        try:
            result = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=3600
            )
            
            if result.returncode == 0:
                print(result.stdout)
                results_eval.append({
                    'method': method,
                    'temperature': temperature,
                    'output': result.stdout
                })
            else:
                print(f"Error: {result.stderr}")
        
        except subprocess.TimeoutExpired:
            print(f"  ⚠️  Timeout (>1 hour)")
        except Exception as e:
            print(f"  ⚠️  Error: {str(e)}")

print("\n" + "="*50)
print("Evaluation completed!")
print("="*50)

# Save evaluation results
if results_eval:
    eval_results_path = Path(CONFIG['output_dir']) / 'evaluation_results.txt'
    with open(eval_results_path, 'w') as f:
        f.write("Fine-tuned Model Evaluation Results\n")
        f.write(f"Checkpoint: {EVAL_CHECKPOINT}\n")
        f.write(f"Training time: {training_time:.1f}h\n\n")
        for r in results_eval:
            f.write(f"\n{r['method']} @ T={r['temperature']}:\n")
            f.write(r['output'])
    print(f"✓ Results saved: {eval_results_path}")


## Save Configuration

Save the configuration for reproducibility.

## Summary & Next Steps

Day 1-2 fine-tuning complete! 


In [ ]:
print("\n" + "="*60)
print("FINE-TUNING COMPLETE - DAY 1-2 SUMMARY")
print("="*60)

summary = f"""
📊 Training Results:
   Epochs: {CONFIG['epochs']}
   Batches per epoch: {len(train_loader) if train_loader else 'N/A'}
   Final loss: {epoch_loss:.4f}
   Best loss: {best_loss:.4f}
   Training time: {training_time:.1f}h
   
✅ Checkpoints saved:
   - eomt_finetuned_best.pth (best loss)
   - eomt_finetuned_epoch*.pth (per epoch)
   - eomt_finetuned_final.pth (final model)
   - training_loss.csv (loss history)
   - training_loss.png (loss plot)
   - config.json (configuration)

📈 Evaluation:
   Method: {EVAL_METHODS[0]}
   Temperature: {EVAL_TEMPERATURES[0]}
   Results saved to: evaluation_results.txt

🎯 Next Steps (Day 3-4):
   1. Implement Outlier Exposure pipeline
   2. Download COCO dataset subset
   3. Create cut-paste augmentation script
   4. Train with both LogitNorm + OE
   5. Compare improvements

📁 Output directory:
   {CONFIG['output_dir']}
"""

print(summary)

# Save summary
summary_path = Path(CONFIG['output_dir']) / 'SUMMARY.txt'
with open(summary_path, 'w') as f:
    f.write(summary)

print(f"✓ Summary saved to: {summary_path}")


In [ ]:
import json

config_path = Path(CONFIG['output_dir']) / 'config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"✓ Configuration saved to {config_path}")